In [201]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-pya0zhtm
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-pya0zhtm
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 28f872a2f99a1b201bcd0db14fdbc5a496b9bfd7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [202]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <iostream> // for std::ostream
#include <vector>

namespace csc485b{
namespace a2{

using node_t = int;
using edge_t = int2;

using edge_list_t = std::vector< edge_t >;
using node_list_t = std::vector< node_t >;

} // namespace a2
} // namespace csc485b


In [203]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [204]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <cassert>  // for assert()
#include <cstddef>  // std::size_t type
#include <random>   // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a2 {

/**
 * Generates and returns a vector of random edges
 * for a graph `G=(V,E)` with `n=|V|=n` and expected `m=|E|`.
 * Referred to as an Erdős-Rényi graph.
 *
 * @see https://networkx.org/documentation/stable/reference/generated/networkx.generators.random_graphs.fast_gnp_random_graph.html#networkx.generators.random_graphs.fast_gnp_random_graph
 */
edge_list_t generate_graph( std::size_t n, std::size_t m )
{
    assert( "At most n(n-1) edges in a simple graph" && m < n * ( n - 1 ) );

    int const probability = ( 100 * m ) / ( n * ( n - 1 ) );

    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20241008;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 100);

    edge_list_t random_edges;
    random_edges.reserve( 2 * m );

    for( node_t u = 0; u < n; ++u )
    {
        for( node_t v = u + 1; v < n; ++v )
        {
            auto const dice_roll = distrib( rng );
            if( dice_roll <= probability )
            {
                random_edges.push_back( make_int2( u, v ) );
                random_edges.push_back( make_int2( v, u ) );
            }
        }
    }

    random_edges.resize( random_edges.size() );


    return random_edges;
}

} // namespace a2
} // namespace csc485b


In [ ]:
%%cuda_group_save -g "source" -n "dense_graph.h"
/**
 * The file in which you will implement your DenseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A DenseGraph is optimised for a graph in which the number of edges
 * is close to n(n-1). It is represented using an adjacency matrix.
 */
struct DenseGraph
{
	std::size_t n; /**< Number of nodes in the graph. */
	node_t * adjacencyMatrix; /** Pointer to an n x n adj. matrix */

	/** Returns number of cells in the adjacency matrix. */
	__device__ __host__ __forceinline__
	std::size_t matrix_size() const { return n * n; }
};


namespace gpu {


/**
 * Constructs a DenseGraph from an input edge list of m edges.
 *
 * @pre The pointers in DenseGraph g have already been allocated.
 */
__global__
void build_graph( DenseGraph g, edge_t const * edge_list, std::size_t m )
{
	int const tid  = threadIdx.y * blockDim.x + threadIdx.x; // 0..1023 inside the block
	std::size_t const th_id = blockIdx.x * ( blockDim.x * blockDim.y ) + tid; // global id
	if ( th_id >= m ) { return; }
	int x = edge_list[ th_id ].x;
	int y = edge_list[ th_id ].y;
	g.adjacencyMatrix[ x * g.n + y ] = 1;
	return;
}

inline void launch_build( DenseGraph g, edge_t const * d_edges, std::size_t m )
{
	dim3 const threads( 32, 32 );
	std::size_t const blocks = ( m + 1023 ) / 1024;
	build_graph<<< blocks, threads >>>( g, d_edges, m );
}

/**
  * Repopulates the adjacency matrix as a new graph that represents
  * the two-hop neighbourhood of input graph g
  */
__global__
void two_hop_reachability( DenseGraph g, node_t * result )
{
	std::size_t const i = blockIdx.y * blockDim.y + threadIdx.y;   // row
	std::size_t const j = blockIdx.x * blockDim.x + threadIdx.x;   // column
	if ( i >= g.n || j >= g.n ) return;

	// square: count paths i -> k -> j
	node_t sum = 0;
	for ( std::size_t k = 0; k < g.n; ++k )
		sum += g.adjacencyMatrix[ i * g.n + k ] * g.adjacencyMatrix[ k * g.n + j ];

	// clamp to 0/1 and remove the diagonal
	result[ i * g.n + j ] = ( sum > 0 && i != j ) ? 1 : 0;
}

__global__
void two_hop_reachability_tile( DenseGraph g, node_t * result)
{
	__shared__ node_t row[ 32 ][ 32 ];
	__shared__ node_t column[ 32 ][ 32 ];
	std::size_t const i = blockIdx.y * blockDim.y + threadIdx.y;   // row
	std::size_t const j = blockIdx.x * blockDim.x + threadIdx.x;   // column
	std::size_t num_of_rounds = ( g.n + 31 ) / 32;
	node_t sum = 0;
	for(std::size_t round = 0; round < num_of_rounds; ++round){
		row[ threadIdx.y ][ threadIdx.x ] = ( i < g.n && round * 32 + threadIdx.x  < g.n ) ? g.adjacencyMatrix[ i * g.n + round * 32 + threadIdx.x ] : 0;
		column[ threadIdx.y ][ threadIdx.x ] = ( j < g.n && round * 32 + threadIdx.y < g.n ) ? g.adjacencyMatrix[ ( round * 32 + threadIdx.y ) * g.n + j ] : 0; 
		__syncthreads();
		for ( int k = 0; k < 32; ++k )
			sum += row[ threadIdx.y ][ k ] * column[ k ][ threadIdx.x ];
		__syncthreads();
	}
	if( i < g.n && j < g.n )
		result[ i * g.n + j ] = ( sum > 0 && i != j ) ? 1 : 0;
}

__global__
void two_hop_reachability_warp( DenseGraph g, node_t * result){
	__shared__ unsigned int rowBits[ 32 ];
	__shared__ unsigned int colBits[ 32 ];
	__shared__ int all_found[ 2 ];
	std::size_t const i = blockIdx.y * blockDim.y + threadIdx.y;   // row
	std::size_t const j = blockIdx.x * blockDim.x + threadIdx.x;   // column
	std::size_t num_of_rounds = ( g.n + 31 ) / 32;
	unsigned int found = 0;
	for (std::size_t round = 0; round < num_of_rounds; ++round){
		unsigned int s = round % 2;
		unsigned int row_val = ( ( blockIdx.y * 32 + threadIdx.y ) < g.n && round * 32 + threadIdx.x < g.n ) ? g.adjacencyMatrix[ ( blockIdx.y * 32 + threadIdx.y ) * g.n + (round * 32 + threadIdx.x) ] : 0;
		unsigned int col_val = ( ( blockIdx.x * 32 + threadIdx.y ) < g.n && round * 32 + threadIdx.x < g.n ) ? g.adjacencyMatrix[ ( blockIdx.x * 32 + threadIdx.y ) * g.n + (round * 32 + threadIdx.x) ] : 0; 
		unsigned int row_val_word = __ballot_sync( 0xFFFFFFFFu, row_val != 0 );
		unsigned int col_val_word = __ballot_sync( 0xFFFFFFFFu, col_val != 0 );
		rowBits[ threadIdx.y ] = row_val_word;
		colBits[ threadIdx.y ] = col_val_word;
		all_found[ s ] = 1;
		__syncthreads();
		// https://www.geeksforgeeks.org/cpp/cpp-bitwise-operators/
		found |= ( rowBits[ threadIdx.y ] & colBits[ threadIdx.x ] ) != 0;
		if ( !found ) all_found[ s ] = 0;
		__syncthreads();
		if ( all_found[ s ] == 1 ) break;
	}
	if( i < g.n && j < g.n )
		result[ i * g.n + j ] = ( found && i != j ) ? 1 : 0;
}


inline void launch_two_hop( DenseGraph g )
{
	std::size_t const bytes = sizeof( node_t ) * g.n * g.n;
	node_t * result;
	cudaMalloc( (void**)&result, bytes );

	dim3 const threads( 32, 32 );
	dim3 const blocks( ( g.n + 31 ) / 32, ( g.n + 31 ) / 32 );   // 2D grid: one thread per cell
	// two_hop_reachability<<< blocks, threads >>>(g, result);
	// two_hop_reachability_tile<<< blocks, threads >>>( g, result );
	two_hop_reachability_warp<<< blocks, threads >>>( g, result );
	cudaMemcpy( g.adjacencyMatrix, result, bytes, cudaMemcpyDeviceToDevice );   // "in place"
	cudaFree( result );
}

} // namespace gpu
} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "sparse_graph.h"
/**
 * The file in which you will implement your SparseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A SparseGraph is optimised for a graph in which the number of edges
 * is close to cn, for a small constanct c. It is represented in CSR format.
 */
struct SparseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  std::size_t m; /**< Number of edges in the graph. */
  node_t * neighbours_start_at; /** Pointer to an n=|V| offset array */
  node_t * neighbours; /** Pointer to an m=|E| array of edge destinations */
};


namespace gpu {

__device__
node_t warp_exclusive_scan( node_t val )
{
    unsigned constexpr FULL_MASK = 0xFFFFFFFFu;
    int const lane = threadIdx.x & 31;
    for( int stride = 1; stride < 32; stride <<= 1 )
    {
        node_t const other = __shfl_up_sync( FULL_MASK, val, stride );
        if( lane >= stride ) val += other;
    }
    node_t const prev = __shfl_up_sync( FULL_MASK, val, 1 );
    return ( lane == 0 ) ? 0 : prev;
}

__device__
node_t block_exclusive_scan( node_t count, node_t & block_total )
{
    __shared__ node_t sum_of_warp_before[ 32 ];
    __shared__ node_t total;
    int const lane = threadIdx.x;
    int const warp = threadIdx.y;
    int const tid  = warp * 32 + lane;

    node_t v = warp_exclusive_scan( count );
    if ( lane == 31 ) sum_of_warp_before[ warp ] = v + count;
    __syncthreads();

    if ( warp == 0 )
    {
        node_t t = sum_of_warp_before[ lane ];
        sum_of_warp_before[ lane ] = warp_exclusive_scan( t );
    }
    __syncthreads();

    v += sum_of_warp_before[ warp ];
    if ( tid == 1023 ) total = v + count; // last thread knows the block total
    __syncthreads();

    block_total = total;
    __syncthreads(); // so the next call can reuse the shared arrays
    return v;
}

// 1. make histogram
__global__
void count_degrees( SparseGraph g, edge_t const * edge_list, std::size_t m )
{
    std::size_t const th_id = blockIdx.x * ( blockDim.x * blockDim.y ) + threadIdx.y * blockDim.x + threadIdx.x;
    if ( th_id < m ) atomicAdd( &g.neighbours_start_at[ edge_list[th_id].x ], 1 );
}

// 2. Exclusive Scan
__global__
void scan_offsets( SparseGraph g, node_t * copy )
{
    int const tid = threadIdx.y * blockDim.x + threadIdx.x;
    node_t carry = 0;
    for ( std::size_t base = 0; base < g.n; base += 1024 )
    {
        std::size_t const i = base + tid;
        node_t count = ( i < g.n ) ? g.neighbours_start_at[ i ] : 0;
        node_t chunk_total;
        node_t v = block_exclusive_scan( count, chunk_total );
        if ( i < g.n )
        {
            g.neighbours_start_at[ i ] = carry + v; // exclusive: start of list
            copy[ i ] = carry + v;
        }
        carry += chunk_total;
    }
}


// 3. many blocks: scatter using copy
__global__
void scatter( SparseGraph g, edge_t const * edge_list, std::size_t m, node_t * copy )
{
    std::size_t const th_id = blockIdx.x * ( blockDim.x * blockDim.y ) + threadIdx.y * blockDim.x + threadIdx.x;
    if ( th_id < m )
    {
        int x = edge_list[ th_id ].x;
        int y = edge_list[ th_id ].y;
        int slot = atomicAdd( &copy[x], 1 );
        g.neighbours[ slot ] = y;
    }
}

inline void launch_build( SparseGraph g, edge_t const * d_edges, std::size_t m )
{
    dim3 const threads( 32, 32 );
    std::size_t const blocks = ( m + 1023 ) / 1024;

    node_t * copy;
    cudaMalloc( (void**)&copy, sizeof( node_t ) * g.n );

    count_degrees <<< blocks, threads >>>( g, d_edges, m );
    scan_offsets <<< 1, threads >>>( g, copy );
    scatter <<< blocks, threads >>>( g, d_edges, m, copy );

    cudaDeviceSynchronize();
    cudaFree( copy );
}



/**
 * Constructs a SparseGraph from an input edge list of m edges.
 *
 * @pre The pointers in SparseGraph g have already been allocated.
 */
// __global__
// void build_graph( SparseGraph g, edge_t const * edge_list, std::size_t m )
// {
//     if ( blockIdx.x != 0 ) return;
    
//     __shared__ node_t copy[ 1024 ];  
//     __shared__ node_t sum_of_warp_before[ 32 ];
//     int const tid  = threadIdx.y * blockDim.x + threadIdx.x;
//     int const lane = threadIdx.x;
//     int const warp = threadIdx.y;  
//     std::size_t const th_id = blockIdx.x * ( blockDim.x * blockDim.y ) + tid;  
//     if ( tid == 0 ) printf( "m = %llu\n", (unsigned long long) g.m );
//     // Make Histogram of value x
//     for ( std::size_t e = tid; e < m; e += 1024 )
//         atomicAdd( &g.neighbours_start_at[ edge_list[e].x ], 1 );
//     __syncthreads();

//     // Exclusive scan
//     node_t count = ( tid < g.n ) ? g.neighbours_start_at[ tid ] : 0;
//     node_t v = warp_exclusive_scan(count);

//     // each warp's last lane saves the warp total
//     if ( lane == 31 ) sum_of_warp_before[ warp ] = v + count;
//     __syncthreads();
 
//     // warp 0 turns the totals into "sum of all warps before me"
//     if ( warp == 0 )
//     {
//         node_t total = sum_of_warp_before[ lane ];
//         sum_of_warp_before[ lane ] = warp_exclusive_scan( total );
//     }
//     __syncthreads();

//     // every thread adds its warp's offset
//     v += sum_of_warp_before[ warp ];

//     if ( tid < g.n ) {
//         g.neighbours_start_at[ tid ] = v;
//         copy[ tid ] = v;
//     }
//     __syncthreads();

//     // if ( th_id < g.n ) g.neighbours_start_at[th_id] = v;          // store it back
//     // copy[th_id] = v;   
//     // __syncwarp();

//     // Make Scatter using histogram
//     for ( std::size_t e = tid; e < m; e += 1024 )
//     {
//         int x = edge_list[ e ].x;
//         int y = edge_list[ e ].y;
//         int slot = atomicAdd( &copy[x], 1 );
//         g.neighbours[slot] = y;
//     }
// }

/**
  * Repopulates the adjacency lists as a new graph that represents
  * the two-hop neighbourhood of input graph g
  */
__global__
void two_hop_reachability( SparseGraph g )
{
    // IMPLEMENT ME!
    // algorithm unknown
    return;
}

inline void launch_two_hop( SparseGraph g )
{
    // not implemented :)
}

} // namespace gpu
} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data,
 * runs the CPU baseline, and then runs your code.
 */

#include <chrono>   // for timing
#include <iostream> // std::cout, std::endl
#include <iterator> // std::ostream_iterator
#include <vector>

#include "dense_graph.h"
#include "sparse_graph.h"

#include "data_generator.h"
#include "data_types.h"

/**
 * Runs timing tests on a CUDA graph implementation.
 * Consists of independently constructing the graph and then
 * modifying it to its two-hop neighbourhood.
 */
template < typename DeviceGraph >
void run( DeviceGraph g, csc485b::a2::edge_t const * d_edges, std::size_t m )
{
    cudaDeviceSynchronize();
    auto const build_start = std::chrono::high_resolution_clock::now();

    csc485b::a2::gpu::launch_build(g, d_edges, m);
    cudaDeviceSynchronize();
    auto const reachability_start = std::chrono::high_resolution_clock::now();

    // neither does this!
    csc485b::a2::gpu::launch_two_hop( g );

    cudaDeviceSynchronize();
    auto const end = std::chrono::high_resolution_clock::now();

    std::cout << "Build time: "
              << std::chrono::duration_cast<std::chrono::microseconds>(reachability_start - build_start).count()
              << " us"
              << std::endl;

    std::cout << "Reachability time: "
              << std::chrono::duration_cast<std::chrono::microseconds>(end - reachability_start).count()
              << " us"
              << std::endl;
}

/**
 * CPU reference for dense two-hop reachability.
 * Returns true if result[i*n + j] == 1 exactly when j is reachable from i
 * by a path of length 2 (i != j). Prints the first mismatch otherwise.
 */
bool check_dense_two_hop( std::vector< csc485b::a2::edge_t > const & edges,
                          std::size_t n,
                          std::vector< csc485b::a2::node_t > const & result )
{
    std::size_t const words = ( n + 63 ) / 64;   // 64 cells per word

    // adjacency matrix as bit rows
    std::vector< unsigned long long > adj( n * words, 0 );
    for ( auto const & e : edges )
        adj[ e.x * words + e.y / 64 ] |= 1ULL << ( e.y % 64 );

    std::vector< unsigned long long > row( words );
    for ( std::size_t i = 0; i < n; ++i )
    {
        // row i of the two-hop = OR of rows k, for every neighbour k of i
        for ( std::size_t w = 0; w < words; ++w ) row[w] = 0;
        for ( std::size_t k = 0; k < n; ++k )
            if ( ( adj[ i * words + k / 64 ] >> ( k % 64 ) ) & 1ULL )
                for ( std::size_t w = 0; w < words; ++w )
                    row[w] |= adj[ k * words + w ];

        row[ i / 64 ] &= ~( 1ULL << ( i % 64 ) );   // remove the diagonal

        // compare with the GPU result
        for ( std::size_t j = 0; j < n; ++j )
        {
            csc485b::a2::node_t const expected = ( row[ j / 64 ] >> ( j % 64 ) ) & 1ULL;
            if ( result[ i * n + j ] != expected )
            {
                std::cout << "first mismatch at (" << i << "," << j << "): expected "
                          << expected << ", got " << result[ i * n + j ] << std::endl;
                return false;
            }
        }
    }
    return true;
}

/**
 * Allocates space for a dense graph and then runs the test code on it.
 */
void run_dense( csc485b::a2::edge_t const * d_edges, std::size_t n, std::size_t m )
{
    using namespace csc485b;

    // allocate device DenseGraph
    a2::node_t * d_matrix;
    cudaMalloc( (void**)&d_matrix, sizeof( a2::node_t ) * n * n );
    cudaMemset( d_matrix, 0, sizeof( a2::node_t ) * n * n );
    a2::DenseGraph d_dg{ n, d_matrix };

    run( d_dg, d_edges, m );

    
    // check output?
    std::vector< a2::node_t > host_matrix( d_dg.matrix_size() );
    a2::DenseGraph dg{ n, host_matrix.data() };
    cudaMemcpy( dg.adjacencyMatrix, d_dg.adjacencyMatrix, sizeof( a2::node_t ) * d_dg.matrix_size(), cudaMemcpyDeviceToHost );
    // std::copy( host_matrix.cbegin(), host_matrix.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );

    // CPU check: matrix must have a 1 exactly where there is an edge
    std::vector< a2::edge_t > h_edges( m );
    cudaMemcpy( h_edges.data(), d_edges, sizeof( a2::edge_t ) * m, cudaMemcpyDeviceToHost );
    // std::vector< a2::node_t > expected_matrix( n * n, 0 );
    // for ( auto const& e : h_edges ) expected_matrix[ e.x * n + e.y ] = 1;
    // std::cout << ( expected_matrix == host_matrix ? "dense build OK" : "dense build WRONG" ) << std::endl;
    
    std::cout << ( check_dense_two_hop( h_edges, n, host_matrix ) ? "dense two-hop OK" : "dense two-hop WRONG" ) << std::endl;
    // clean up
    cudaFree( d_matrix );
}

/**
 * Allocates space for a sparse graph and then runs the test code on it.
 */
void run_sparse( csc485b::a2::edge_t const * d_edges, std::size_t n, std::size_t m )
{
    using namespace csc485b;

    // allocate device SparseGraph
    a2::node_t * d_offsets, * d_neighbours;
    cudaMalloc( (void**)&d_offsets,    sizeof( a2::node_t ) * n );
    cudaMemset( d_offsets, 0, sizeof( a2::node_t ) * n );
    cudaMalloc( (void**)&d_neighbours, sizeof( a2::node_t ) * m );
    a2::SparseGraph d_sg{ n, m, d_offsets, d_neighbours };

    run( d_sg, d_edges, m );

    // copy results back to host
    std::vector< a2::node_t > host_offsets( n );
    std::vector< a2::node_t > host_neighbours( m );
    cudaMemcpy( host_offsets.data(),    d_offsets,    sizeof( a2::node_t ) * n, cudaMemcpyDeviceToHost );
    cudaMemcpy( host_neighbours.data(), d_neighbours, sizeof( a2::node_t ) * m, cudaMemcpyDeviceToHost );

    // CPU check: count degrees, prefix-sum, compare
    std::vector< a2::node_t > expected( n, 0 );
    std::vector< a2::edge_t > h_edges( m );
    cudaMemcpy( h_edges.data(), d_edges, sizeof( a2::edge_t ) * m, cudaMemcpyDeviceToHost );
    for ( auto const& e : h_edges ) expected[ e.x ]++;
    a2::node_t run_sum = 0;
    for ( std::size_t i = 0; i < n; ++i ) { a2::node_t c = expected[i]; expected[i] = run_sum; run_sum += c; }
    std::cout << ( expected == host_offsets ? "offsets OK" : "offsets WRONG" ) << std::endl;


    // print them
    // std::cout << "\nneighbours_start_at: ";
    // std::copy( host_offsets.cbegin(), host_offsets.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );
    // std::cout << "\nneighbours: ";
    // std::copy( host_neighbours.cbegin(), host_neighbours.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );
    // std::cout << std::endl;
    
    // clean up
    cudaFree( d_neighbours );
    cudaFree( d_offsets );
}

int main()
{
    using namespace csc485b;

    // Create input
    std::size_t constexpr n = 4096;
    std::size_t constexpr expected_degree = n >> 1;

    a2::edge_list_t const graph = a2::generate_graph( n, n * expected_degree );
    std::size_t const m = graph.size();

    // lazily echo out input graph
    // for( auto const& e : graph )
    // {
    //     std::cout << "(" << e.x << "," << e.y << ") ";
    // }

    // allocate and memcpy input to device
    a2::edge_t * d_edges;
    cudaMalloc( (void**)&d_edges, sizeof( a2::edge_t ) * m );
    cudaMemcpyAsync( d_edges, graph.data(), sizeof( a2::edge_t ) * m, cudaMemcpyHostToDevice );

    // run your code!
    run_dense ( d_edges, n, m );
    cudaDeviceSynchronize();
    run_sparse( d_edges, n, m );

    return EXIT_SUCCESS;
}

In [208]:
%cuda_group_run --group "source" --compiler-args "-O3 -g -std=c++20 -arch=sm_75"

(0,2) (2,0) (0,3) (3,0) (0,4) (4,0) (0,5) (5,0) (0,6) (6,0) (1,3) (3,1) (1,4) (4,1) (1,5) (5,1) (1,7) (7,1) (2,3) (3,2) (2,5) (5,2) (3,4) (4,3) (3,5) (5,3) (3,6) (6,3) (4,5) (5,4) (4,6) (6,4) (4,7) (7,4) (5,6) (6,5) (5,7) (7,5) (6,7) (7,6) Build time: 2 us
Reachability time: 0 us
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 Build time: 0 us
Reachability time: 0 us

